# Prediction tutorial: choose any energy target

The root `prediction` package forecasts an explicitly selected energy value observed over time. Consumption/demand and production/generation are equal examples of the same workflow: choose a target column, train a predictor, and receive output named for that target.

## Terms and workflow

- **Timestamp:** when an observation was measured.
- **Target:** the numeric energy value to forecast, such as `Consumption` or `Production`.
- **Feature:** a numeric input available when a forecast is made. Here, calendar features describe repeating hour, weekday, and seasonal patterns.
- **Model:** the fitted relationship between features and the selected target.
- **Prediction:** an estimate of that target for a requested timestamp.

| Stage | Input | Output |
|---|---|---|
| Prepare | timestamp + target rows | cleaned chronological data |
| Describe time | timestamps | numeric calendar features |
| Train | features + selected target | fitted model |
| Predict | a future timestamp | forecast target value |

## 1. Create peer demand and generation datasets

Each CSV needs a `Datetime` column and the numeric target named by `target_col`. The synthetic demand and generation frames below use identical timestamps so the examples differ only in the selected target.

In [ ]:
from pathlib import Path
import tempfile

import numpy as np
import pandas as pd

from prediction.energy_predictor import (
    create_energy_predictor,
    generate_day_predictions,
    predict_energy,
    prediction_column_name,
)

temporary_workspace = tempfile.TemporaryDirectory()
temporary_path = Path(temporary_workspace.name)

In [ ]:
timestamps = pd.date_range("2025-01-01", periods=14 * 24, freq="h")
hour = timestamps.hour.to_numpy()
weekday = timestamps.weekday.to_numpy()

consumption_data = pd.DataFrame({
    "Datetime": timestamps,
    "Consumption": 24 + 5 * np.sin(2 * np.pi * (hour - 7) / 24) + 2 * (weekday < 5),
})
production_data = pd.DataFrame({
    "Datetime": timestamps,
    "Production": 18 * np.maximum(0, np.sin(np.pi * (hour - 6) / 12)),
})

datasets = {
    "demand": (consumption_data, "Consumption"),
    "generation": (production_data, "Production"),
}
csv_paths = {}
for label, (frame, target_col) in datasets.items():
    csv_path = temporary_path / f"{label}.csv"
    frame.to_csv(csv_path, index=False)
    csv_paths[label] = csv_path

pd.concat({label: frame.head(3) for label, (frame, _) in datasets.items()})

## 2. Train both targets with the primary factory

`create_energy_predictor` requires `target_col`, so the target choice is visible in every primary workflow. It performs loading, time-feature creation, feature detection, and model training.

In [ ]:
predictors = {}
for label, (_, target_col) in datasets.items():
    predictors[label] = create_energy_predictor(
        csv_paths[label],
        target_col=target_col,
    )

The predictor object uses the same methods for either target. A single timestamp returns a numeric estimate, while `predict_days` returns a table whose prediction column is derived from the selected target.

In [ ]:
daily_forecasts = {}
for label, (_, target_col) in datasets.items():
    predictor = predictors[label]
    value = predictor.predict("2025-01-15", "12:00")
    daily_forecasts[label] = predictor.predict_days(
        "2025-01-15",
        num_days=1,
    )
    output_col = prediction_column_name(target_col)
    print(f"{label.title()} at 12:00: {value:.2f}; output column: {output_col}")

In [ ]:
demand_output = daily_forecasts["demand"][[
    "Date", "Time", "Predicted_Consumption"
]].iloc[10:15]
generation_output = daily_forecasts["generation"][[
    "Date", "Time", "Predicted_Production"
]].iloc[10:15]

demand_output, generation_output

## 3. Use the generic prediction function

`predict_energy` is the primary lower-level API when a model and its feature configuration are already available. `generate_day_predictions` likewise accepts an explicit target so its output remains target-specific.

In [ ]:
for label, (_, target_col) in datasets.items():
    predictor = predictors[label]
    value = predict_energy(
        predictor.model,
        predictor.feature_cols,
        predictor.feature_engineering_fn,
        "2025-01-15",
        "08:00",
    )
    day = generate_day_predictions(
        predictor.model,
        predictor.feature_cols,
        predictor.feature_engineering_fn,
        start_date="2025-01-15",
        target_col=target_col,
    )
    print(f"{label.title()} at 08:00: {value:.2f}; columns: {list(day.columns)}")

## 4. Apply the workflow to real data

1. Replace the temporary CSV with timestamped measurements.
2. Pass the exact demand, generation, net-load, or other energy column as `target_col`.
3. Add only features available at forecast time, such as calendar values or weather forecasts.
4. Evaluate candidate models on later, unseen timestamps before operational use.
5. Retrain on an appropriate recent history before producing forecasts.

In [ ]:
temporary_workspace.cleanup()

## Backward compatibility

Existing callers may still import `prediction.predicting_consumption_model` or call deprecated names such as `create_predictor` and `predict_consumption`. Those paths retain consumption defaults and emit deprecation warnings; new code should import `prediction.energy_predictor`, choose `target_col` explicitly, and use `create_energy_predictor` or `predict_energy`.